In [2]:
import sys
from pathlib import Path
import pandas as pd

# so the notebook can import from src/
sys.path.append(str(Path.cwd().parent))

from src.database.db_connection import get_connection, load_config, PROJECT_ROOT

config = load_config()
df = pd.read_csv(PROJECT_ROOT / config["paths"]["cleaned_data"])

# SQL-friendly column names: lowercase, no spaces
df.columns = (df.columns.str.strip()
                        .str.lower()
                        .str.replace(r"[^a-z0-9]+", "_", regex=True)
                        .str.strip("_"))

conn = get_connection()
df.to_sql("orders", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM orders", conn))
print(pd.read_sql("PRAGMA table_info(orders)", conn)[["name", "type"]])
conn.close()

   total_rows
0      172762
                           name     type
0                          type     TEXT
1        days_for_shipping_real  INTEGER
2   days_for_shipment_scheduled  INTEGER
3             benefit_per_order     REAL
4            sales_per_customer     REAL
5               delivery_status     TEXT
6            late_delivery_risk  INTEGER
7                   category_id  INTEGER
8                 category_name     TEXT
9                 customer_city     TEXT
10             customer_country     TEXT
11                  customer_id  INTEGER
12             customer_segment     TEXT
13               customer_state     TEXT
14             customer_zipcode     REAL
15                department_id  INTEGER
16              department_name     TEXT
17                     latitude     REAL
18                    longitude     REAL
19                       market     TEXT
20                   order_city     TEXT
21                order_country     TEXT
22            order_customer_

In [3]:
from src.database.db_connection import get_connection
import pandas as pd
conn = get_connection()
def run(sql):
    return pd.read_sql(sql, conn)

In [4]:
#Query 1: SELECT, WHERE, ORDER BY, LIMIT
run("""
SELECT order_id, shipping_mode, order_region, shipping_gap
FROM orders
WHERE shipping_mode = 'Standard Class' AND late_delivery_risk = 1
ORDER BY shipping_gap DESC
LIMIT 10
""")

,order_id,shipping_mode,order_region,shipping_gap
0,75910,Standard Class,Eastern Asia,2
1,75895,Standard Class,South Asia,2
2,52640,Standard Class,South America,2
3,6245,Standard Class,South America,2
4,54585,Standard Class,South America,2
5,52640,Standard Class,South America,2
6,58375,Standard Class,Central America,2
7,7980,Standard Class,Central America,2
8,57185,Standard Class,Central America,2
9,51725,Standard Class,Central America,2


In [5]:
#Query 2: GROUP BY with one category
run("""
SELECT shipping_mode,
       COUNT(*) AS total_orders,
       ROUND(AVG(late_delivery_risk) * 100, 1) AS late_pct,
       ROUND(AVG(shipping_gap), 2) AS avg_gap_days
FROM orders
GROUP BY shipping_mode
ORDER BY late_pct DESC
""")

,shipping_mode,total_orders,late_pct,avg_gap_days
0,First Class,26512,100.0,1.00
1,Second Class,33806,79.8,1.99
2,Same Day,9293,47.9,0.48
3,Standard Class,103151,39.8,-0.01


In [6]:
#Query 3: GROUP BY with two columns
run("""
SELECT order_region, shipping_mode,
       COUNT(*) AS total_orders,
       ROUND(AVG(late_delivery_risk) * 100, 1) AS late_pct
FROM orders
GROUP BY order_region, shipping_mode
HAVING COUNT(*) > 500
ORDER BY late_pct DESC
LIMIT 10
""")

,order_region,shipping_mode,total_orders,late_pct
0,Caribbean,First Class,1132,100.0
1,Central America,First Class,4185,100.0
2,East of USA,First Class,1050,100.0
3,Eastern Asia,First Class,989,100.0
4,Eastern Europe,First Class,632,100.0
5,Northern Europe,First Class,1361,100.0
6,Oceania,First Class,1270,100.0
7,South America,First Class,2154,100.0
8,South Asia,First Class,1240,100.0
9,South of USA,First Class,571,100.0


In [7]:
#Query 4: The ranking query for the chunk deliverable
run("""
SELECT shipping_mode,
       COUNT(*) AS total_orders,
       ROUND(AVG(late_delivery_risk) * 100, 1) AS late_pct,
       ROUND(AVG(days_for_shipping_real), 2) AS avg_real_days,
       ROUND(AVG(days_for_shipment_scheduled), 2) AS avg_scheduled_days,
       RANK() OVER (ORDER BY AVG(late_delivery_risk) ASC) AS reliability_rank
FROM orders
GROUP BY shipping_mode
ORDER BY reliability_rank
""")

,shipping_mode,total_orders,late_pct,avg_real_days,avg_scheduled_days,reliability_rank
0,Standard Class,103151,39.8,3.99,4.0,1
1,Same Day,9293,47.9,0.48,0.0,2
2,Second Class,33806,79.8,3.99,2.0,3
3,First Class,26512,100.0,2.00,1.0,4


In [8]:
df.groupby("shipping_mode")["late_delivery_risk"].mean() * 100

shipping_mode
First Class       100.000000
Same Day           47.928548
Second Class       79.829024
Standard Class     39.769852
Name: late_delivery_risk, dtype: float64

In [9]:
from src.database.queries import mode_performance, region_mode_risk, filter_orders

print(mode_performance())
print(region_mode_risk().head())
print(filter_orders(shipping_mode="Same Day", region="Western Europe", limit=5))

    shipping_mode  total_orders  late_pct  avg_real_days  avg_scheduled_days  \
0  Standard Class        103151      39.8           3.99                 4.0   
1        Same Day          9293      47.9           0.48                 0.0   
2    Second Class         33806      79.8           3.99                 2.0   
3     First Class         26512     100.0           2.00                 1.0   

   reliability_rank  
0                 1  
1                 2  
2                 3  
3                 4  
      order_region shipping_mode  total_orders  late_pct
0        Caribbean   First Class          1132     100.0
1  Central America   First Class          4185     100.0
2      East of USA   First Class          1050     100.0
3     Eastern Asia   First Class           989     100.0
4   Eastern Europe   First Class           632     100.0
   order_id shipping_mode    order_region         category_name  shipping_gap  \
0     11825      Same Day  Western Europe                Hockey   